# Финальная сверка Python- и R-результатов

Цель notebook — сравнить итоговые таблицы, полученные в Python и R, и подготовить короткое решение аналитика.

## 1. Подготовка окружения

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd()
output_dir = project_root / 'outputs'

print('Корень проекта:', project_root)
print('Папка outputs:', output_dir)

## 2. Проверка файлов

In [ ]:
required_files = [
    output_dir / 'sales_datamart.csv',
    output_dir / 'r_expected_sales_datamart.csv',
    output_dir / 'report_by_region_channel.csv',
    output_dir / 'r_expected_report_by_region_channel.csv',
]

for file_path in required_files:
    print(('OK: ' if file_path.exists() else 'НЕ НАЙДЕН: ') + str(file_path))

## 3. Загрузка результатов

In [ ]:
python_datamart = pd.read_csv(output_dir / 'sales_datamart.csv')
r_datamart = pd.read_csv(output_dir / 'r_expected_sales_datamart.csv')
python_report = pd.read_csv(output_dir / 'report_by_region_channel.csv')
r_report = pd.read_csv(output_dir / 'r_expected_report_by_region_channel.csv')

print('Python-витрина:', python_datamart.shape)
print('R-витрина:', r_datamart.shape)
print('Python-сводка:', python_report.shape)
print('R-сводка:', r_report.shape)

## 4. Сортировка перед сравнением

Перед сравнением таблицы нужно привести к одному порядку строк. Иначе одинаковые данные могут выглядеть как разные.

In [ ]:
python_datamart = python_datamart.sort_values('order_id').reset_index(drop=True)
r_datamart = r_datamart.sort_values('order_id').reset_index(drop=True)
python_report = python_report.sort_values(['macro_region', 'region_name', 'channel']).reset_index(drop=True)
r_report = r_report.sort_values(['macro_region', 'region_name', 'channel']).reset_index(drop=True)

## 5. Контрольные показатели

In [ ]:
checks = []

def add_check(name, python_value, r_value, tolerance=0.0, comment=''):
    try:
        diff = float(python_value) - float(r_value)
        passed = abs(diff) <= tolerance
    except Exception:
        diff = ''
        passed = python_value == r_value
    checks.append({
        'check_name': name,
        'python_value': python_value,
        'r_value': r_value,
        'difference': diff,
        'tolerance': tolerance,
        'status': 'pass' if passed else 'fail',
        'comment': comment,
    })

add_check('datamart_rows', len(python_datamart), len(r_datamart), comment='Количество строк в витрине')
add_check('datamart_columns', python_datamart.shape[1], r_datamart.shape[1], comment='Количество столбцов в витрине')
add_check('report_rows', len(python_report), len(r_report), comment='Количество строк в региональной сводке')
add_check('total_revenue', round(python_datamart['revenue'].sum(), 2), round(r_datamart['revenue'].sum(), 2), tolerance=0.01, comment='Итоговая выручка')
add_check('total_margin', round(python_datamart['margin'].sum(), 2), round(r_datamart['margin'].sum(), 2), tolerance=0.01, comment='Итоговая маржа')
add_check('unique_orders', python_datamart['order_id'].nunique(), r_datamart['order_id'].nunique(), comment='Уникальные заказы')
add_check('unique_channels', ', '.join(sorted(python_datamart['channel'].unique())), ', '.join(sorted(r_datamart['channel'].unique())), comment='Каналы после нормализации')
add_check('unique_categories', ', '.join(sorted(python_datamart['category'].unique())), ', '.join(sorted(r_datamart['category'].unique())), comment='Категории после нормализации')

comparison = pd.DataFrame(checks)
display(comparison)

## 6. Проверка сводки по регионам и каналам

In [ ]:
numeric_cols = ['orders_count', 'total_quantity', 'total_revenue', 'avg_order_revenue', 'total_margin']
diff_rows = []

for i, (prow, rrow) in enumerate(zip(python_report.to_dict('records'), r_report.to_dict('records')), start=1):
    key = f"{prow['macro_region']} | {prow['region_name']} | {prow['channel']}"
    for col in numeric_cols:
        diff = float(prow[col]) - float(rrow[col])
        if abs(diff) > 0.01:
            diff_rows.append({
                'row_number': i,
                'key': key,
                'column': col,
                'python_value': prow[col],
                'r_value': rrow[col],
                'difference': diff,
            })

report_differences = pd.DataFrame(diff_rows, columns=['row_number', 'key', 'column', 'python_value', 'r_value', 'difference'])
display(report_differences)

if report_differences.empty:
    print('Расхождений в региональной сводке не найдено.')

## 7. Сохранение результата

In [ ]:
comparison.to_csv(output_dir / 'final_python_r_comparison.csv', index=False)
report_differences.to_csv(output_dir / 'final_report_differences.csv', index=False)

print('Сохранено:', output_dir / 'final_python_r_comparison.csv')
print('Сохранено:', output_dir / 'final_report_differences.csv')

## 8. Итоговый вывод аналитика

Заполните вывод своими словами. Используйте фактические контрольные значения из таблицы сверки.

```text
Python- и R-результаты совпали / не совпали по ключевым показателям.

Основные контрольные значения:
- строк в витрине: ...
- итоговая выручка: ...
- итоговая маржа: ...
- строк в региональной сводке: ...

Вывод:
...

Ограничение анализа:
...
```